In [ ]:
%%spark

# ============================================================
# 1_UTILS - Funcoes tecnicas compartilhadas do pipeline
# ============================================================

from datetime import date, datetime
from decimal import Decimal
import re
import uuid

from pyspark.sql import DataFrame

PADRAO_IDENTIFICADOR_SQL = r"^[A-Za-z0-9_]+$"

def validar_hoje_execucao(valor_hoje: str) -> str:
    valor = (valor_hoje or "").strip()

    if not valor:
        raise ErroOperacional(
            codigo="CONFIG_DATA_EXECUCAO_AUSENTE",
            mensagem="Data oficial da execucao nao informada.",
            objeto="HOJE",
            acao="Configurar HOJE no ambiente da sessao Spark antes de executar a rotina.",
        )

    candidatos = [
        valor,
        valor.replace("T", " "),
    ]

    formatos = [
        "%Y-%m-%d",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M:%S.%f",
    ]

    for candidato in candidatos:
        for formato in formatos:
            try:
                return datetime.strptime(candidato[:26], formato).date().isoformat()
            except ValueError:
                pass

    raise ErroOperacional(
        codigo="CONFIG_DATA_EXECUCAO_INVALIDA",
        mensagem="Data oficial da execucao possui formato invalido.",
        objeto="HOJE",
        detalhes={"formato_esperado": "yyyy-MM-dd ou timestamp ISO"},
        acao="Corrigir HOJE no ambiente da sessao Spark.",
    )


def validar_identificador_sql(valor: str, contexto: str) -> str:
    identificador = (valor or "").strip()

    if not identificador:
        raise ValueError(f"{contexto}: identificador SQL vazio.")

    if not re.fullmatch(PADRAO_IDENTIFICADOR_SQL, identificador):
        raise ValueError(f"{contexto}: identificador SQL invalido: {identificador}")

    return identificador


def literal_sql(valor) -> str:
    if valor is None:
        return "NULL"

    if isinstance(valor, bool):
        return "TRUE" if valor else "FALSE"

    if isinstance(valor, Decimal):
        return format(valor, "f")

    if isinstance(valor, (int, float)):
        return str(valor)

    if isinstance(valor, datetime):
        return "'" + valor.strftime("%Y-%m-%d %H:%M:%S.%f").rstrip("0").rstrip(".") + "'"

    if isinstance(valor, date):
        return "'" + valor.isoformat() + "'"

    return "'" + str(valor).replace("'", "''") + "'"


def tipo_sql(tipo_coluna) -> str:
    if hasattr(tipo_coluna, "simpleString"):
        return tipo_coluna.simpleString()

    return str(tipo_coluna)


def literal_sql_tipado(valor, tipo_coluna) -> str:
    return f"CAST({literal_sql(valor)} AS {tipo_sql(tipo_coluna)})"


def coluna_sql(nome_coluna: str, alias: str = None) -> str:
    coluna = f"`{validar_identificador_sql(nome_coluna, 'coluna')}`"

    if alias:
        return f"{validar_identificador_sql(alias, 'alias')}.{coluna}"

    return coluna


def lista_literais_sql(valores: list) -> str:
    return ", ".join(literal_sql(valor) for valor in valores)


def nome_tabela_hive(database: str, tabela: str) -> str:
    database_final = validar_identificador_sql(database, "database")
    tabela_final = validar_identificador_sql(tabela, "tabela")
    return f"{database_final}.{tabela_final}"


def nome_tabela_hive_sql(database: str, tabela: str) -> str:
    database_final = validar_identificador_sql(database, "database")
    tabela_final = validar_identificador_sql(tabela, "tabela")
    return f"`{database_final}`.`{tabela_final}`"


def nome_qualificado_sql(schema: str, tabela: str) -> str:
    return (
        f"{validar_identificador_sql(schema, 'schema')}."
        f"{validar_identificador_sql(tabela, 'tabela')}"
    )


def nome_qualificado_spark_sql(schema: str, tabela: str) -> str:
    schema_final = validar_identificador_sql(schema, "schema")
    tabela_final = validar_identificador_sql(tabela, "tabela")
    return f"`{schema_final}`.`{tabela_final}`"


def spark_sql(query: str) -> DataFrame:
    sql = (query or "").strip()

    if not sql:
        raise ValueError("SQL nao pode ser vazio.")

    return spark.sql(sql)


def _nome_view_sql(prefixo: str) -> str:
    prefixo_limpo = validar_identificador_sql(prefixo or "tmp_sql", "prefixo_view")
    return f"{prefixo_limpo}_{uuid.uuid4().hex}"


def registrar_view_sql(df: DataFrame, prefixo: str = "tmp_sql") -> str:
    if df is None:
        raise ValueError("DataFrame nao pode ser None para registrar view SQL.")

    nome_view = _nome_view_sql(prefixo)
    # EXCECAO_SQL: ponte tecnica para consultar por SQL um DataFrame recebido de outra etapa.
    df.createOrReplaceTempView(nome_view)
    return nome_view


def coletar_sql(query: str):
    df = spark_sql(query)
    # EXCECAO_SQL: coleta pequena no driver usada apenas para validacao, decisao ou estatistica.
    return df.collect()


def coletar_dataframe_sql(df: DataFrame, contexto: str = "COLETA"):
    view = registrar_view_sql(df, f"coleta_{contexto.lower()}")
    return coletar_sql(f"SELECT * FROM {view}")


def existe_sql(query: str) -> bool:
    view = _nome_view_sql("existe_sql")
    spark_sql(f"CREATE OR REPLACE TEMPORARY VIEW {view} AS {query}")
    return bool(coletar_sql(f"SELECT 1 AS OK FROM {view} LIMIT 1"))


def existe_dataframe_sql(df: DataFrame, contexto: str = "EXISTE") -> bool:
    view = registrar_view_sql(df, f"existe_{contexto.lower()}")
    return existe_sql(f"SELECT 1 AS OK FROM {view}")


def contar_dataframe_sql(df: DataFrame, contexto: str = "CONTAGEM") -> int:
    view = registrar_view_sql(df, f"contagem_{contexto.lower()}")
    linhas = coletar_sql(f"SELECT COUNT(1) AS QTD FROM {view}")
    return int(linhas[0]["QTD"])


def normalizar_colunas_hive(df: DataFrame, objeto: str) -> DataFrame:
    if df is None:
        raise ErroContratoDados(
            codigo="HIVE_DATAFRAME_AUSENTE",
            mensagem="Leitura Hive nao retornou um DataFrame.",
            objeto=objeto,
            acao="Verificar a leitura da tabela Hive no ambiente da execucao.",
        )

    colunas_originais = list(df.columns)
    colunas_normalizadas = [str(coluna).strip().upper() for coluna in colunas_originais]
    origens_por_normalizada = {}

    for original, normalizada in zip(colunas_originais, colunas_normalizadas):
        origens_por_normalizada.setdefault(normalizada, []).append(original)

    ambiguidades = {
        normalizada: origens
        for normalizada, origens in origens_por_normalizada.items()
        if len(origens) > 1
    }

    if ambiguidades:
        raise ErroContratoDados(
            codigo="HIVE_COLUNAS_AMBIGUAS",
            mensagem="Tabela Hive possui colunas ambiguas apos normalizacao dos nomes.",
            objeto=objeto,
            detalhes={"colunas_ambiguas": ambiguidades},
            acao="Corrigir o DDL para que cada coluna possua um nome unico sem diferenca apenas de caixa.",
        )

    if colunas_originais == colunas_normalizadas:
        return df

    return df.toDF(*colunas_normalizadas)


def ler_tabela_hive(database: str, tabela: str) -> DataFrame:
    objeto = nome_tabela_hive(database, tabela)

    try:
        df = spark_sql(f"SELECT * FROM {nome_tabela_hive_sql(database, tabela)}")
        return normalizar_colunas_hive(df, objeto)
    except ErroPipeline:
        raise
    except Exception as exc:
        raise ErroAcessoDados(
            codigo="HIVE_LEITURA_FALHOU",
            mensagem="Falha ao ler tabela Hive.",
            objeto=objeto,
            acao="Verificar existencia, permissao, catalogo e disponibilidade da tabela Hive.",
        ) from exc


def tabela_hive_existe(database: str, tabela: str) -> bool:
    database_final = validar_identificador_sql(database, "database")
    tabela_final = validar_identificador_sql(tabela, "tabela")
    objeto = f"{database_final}.{tabela_final}"

    try:
        return bool(spark.catalog.tableExists(tabela_final, database_final))
    except Exception as exc:
        raise ErroAcessoDados(
            codigo="HIVE_VERIFICACAO_EXISTENCIA_FALHOU",
            mensagem="Falha ao verificar existencia da tabela Hive.",
            objeto=objeto,
            acao="Verificar permissao e disponibilidade do catalogo Hive.",
        ) from exc


def ler_parquet_sql(path_hdfs: str) -> DataFrame:
    path = (path_hdfs or "").strip()

    if not path:
        raise ValueError("path_hdfs nao pode ser vazio.")

    if "`" in path:
        raise ValueError(f"path_hdfs invalido para SQL: {path_hdfs}")

    try:
        return spark_sql(f"SELECT * FROM parquet.`{path}`")
    except ErroPipeline:
        raise
    except Exception as exc:
        raise ErroAcessoDados(
            codigo="HDFS_LEITURA_TEMPORARIO_FALHOU",
            mensagem="Falha ao ler arquivo temporario HDFS.",
            objeto=path,
            detalhes={"operacao": "LEITURA_PARQUET"},
            acao="Verificar existencia, permissao e integridade do arquivo temporario HDFS.",
        ) from exc


def path_hdfs_existe(path_hdfs: str) -> bool:
    if not path_hdfs or not str(path_hdfs).strip():
        raise ValueError("path_hdfs nao pode ser vazio.")

    # EXCECAO_SQL: API Hadoop e obrigatoria para existencia fisica de path HDFS.
    try:
        conf = spark._jsc.hadoopConfiguration()
        path = spark._jvm.org.apache.hadoop.fs.Path(path_hdfs)
        fs = path.getFileSystem(conf)
        return fs.exists(path)
    except ErroPipeline:
        raise
    except Exception as exc:
        raise ErroAcessoDados(
            codigo="HDFS_VERIFICACAO_EXISTENCIA_FALHOU",
            mensagem="Falha ao verificar existencia de path HDFS.",
            objeto=str(path_hdfs),
            detalhes={"operacao": "VERIFICAR_EXISTENCIA"},
            acao="Verificar acesso e disponibilidade do HDFS.",
        ) from exc


def remover_path_hdfs_se_existir(
    path_hdfs: str,
    executar: bool,
    logger_etapa=None,
    contexto: str = "HDFS",
) -> bool:
    existe = path_hdfs_existe(path_hdfs)

    if not existe:
        logger_etapa.info(f"[{contexto}][HDFS] Path inexistente; limpeza ja concluida. path={path_hdfs}")
        return True

    if not executar:
        logger_etapa.info(
            f"[{contexto}][DRY_RUN] Path seria removido se executar=True. path={path_hdfs}",
        )
        return False

    # EXCECAO_SQL: API Hadoop e obrigatoria para remocao fisica de path HDFS.
    try:
        conf = spark._jsc.hadoopConfiguration()
        path = spark._jvm.org.apache.hadoop.fs.Path(path_hdfs)
        fs = path.getFileSystem(conf)
        removido = bool(fs.delete(path, True))
    except ErroPipeline:
        raise
    except Exception as exc:
        raise ErroAcessoDados(
            codigo="HDFS_LIMPEZA_TEMPORARIOS_FALHOU",
            mensagem="Falha ao remover path temporario HDFS.",
            objeto=str(path_hdfs),
            detalhes={"operacao": "DELETE_RECURSIVO"},
            acao="Verificar permissao e disponibilidade do path temporario HDFS.",
        ) from exc

    if removido:
        logger_etapa.info(f"[{contexto}][HDFS] Path removido. path={path_hdfs}")
    else:
        logger_etapa.error(f"[{contexto}][HDFS] Falha ao remover path. path={path_hdfs}")

    return removido


def caminhos_temporarios_ctl(path_save_hdfs: str) -> dict:
    tabela_ctl = TABELAS_HIVE["hive_controle_operacional_recomendacao"]
    nomes = CONTRATO_HIVE[tabela_ctl]["nome_temporario"]
    raiz = (path_save_hdfs or "").rstrip("/")

    if not raiz:
        raise ValueError("path_save_hdfs nao pode ser vazio.")

    return {
        "temp_backup": f"{raiz}/{nomes['temp_backup']}",
        "temp_lineage": f"{raiz}/{nomes['temp_lineage']}",
    }


def colunas_df(df: DataFrame) -> list[str]:
    return list(df.columns)


def mapa_colunas_case_insensitive(df: DataFrame) -> dict:
    return {c.strip().lower(): c for c in df.columns}


def resolver_coluna_case_insensitive(df: DataFrame, nomes_candidatos: list[str]):
    mapa = mapa_colunas_case_insensitive(df)

    for nome in nomes_candidatos:
        chave = str(nome).strip().lower()
        if chave in mapa:
            return mapa[chave]

    return None


def exigir_colunas(df: DataFrame, colunas: list[str], contexto: str) -> None:
    mapa = mapa_colunas_case_insensitive(df)
    faltantes = [
        coluna
        for coluna in colunas
        if str(coluna).strip().lower() not in mapa
    ]

    if faltantes:
        raise ErroContratoDados(
            codigo="CONTRATO_COLUNAS_AUSENTES",
            mensagem=f"{contexto}: contrato possui colunas ausentes.",
            detalhes={"colunas_ausentes": faltantes},
            acao="Comparar o schema fisico com o contrato esperado pelo pipeline.",
        )


def nomes_campos_contrato_generico(contrato: dict) -> list[str]:
    return [nome_coluna for nome_coluna, _, _ in contrato["campos"]]


def tipo_campo_contrato_generico(contrato: dict, coluna: str):
    for nome_coluna, tipo_coluna, _ in contrato["campos"]:
        if nome_coluna == coluna:
            return tipo_coluna

    raise ErroContratoDados(
        codigo="CONTRATO_CAMPO_NAO_DEFINIDO",
        mensagem=f"Coluna {coluna} ausente no contrato.",
        detalhes={"coluna": coluna},
        acao="Incluir a coluna no contrato ou corrigir a referencia utilizada pelo pipeline.",
    )


def tipo_campo_contrato_hive(tabela: str, coluna: str):
    return tipo_campo_contrato_generico(CONTRATO_HIVE[tabela], coluna)


# def _select_contrato_sql(view: str, contrato: dict, colunas_origem: list[str]) -> str:
#     origem = set(colunas_origem)
#     expressoes = []

#     for nome_coluna, tipo_coluna, _ in contrato["campos"]:
#         if nome_coluna in origem:
#             expr = f"CAST({coluna_sql(nome_coluna)} AS {tipo_sql(tipo_coluna)})"
#         else:
#             expr = f"CAST(NULL AS {tipo_sql(tipo_coluna)})"

#         expressoes.append(f"{expr} AS {coluna_sql(nome_coluna)}")

#     return f"SELECT {', '.join(expressoes)} FROM {view}"

def _select_contrato_sql(
    view: str,
    contrato: dict,
    colunas_origem: list[str]
) -> str:

    origem = {
        str(coluna).strip().lower(): coluna
        for coluna in colunas_origem
    }

    expressoes = []

    for nome_coluna, tipo_coluna, _ in contrato["campos"]:

        tipo_destino = tipo_sql(tipo_coluna)

        coluna_origem = origem.get(nome_coluna.lower())

        if coluna_origem is not None:

            # NORMALIZA STRINGS
            if tipo_destino.upper() == "STRING":
                expr = (
                    f"TRIM(CAST({coluna_sql(coluna_origem)} AS STRING))"
                )
            else:
                expr = (
                    f"CAST({coluna_sql(coluna_origem)} AS {tipo_destino})"
                )

        else:

            expr = f"CAST(NULL AS {tipo_destino})"

        expressoes.append(
            f"{expr} AS {coluna_sql(nome_coluna)}"
        )

    return f"""
        SELECT
            {', '.join(expressoes)}
        FROM {view}
    """


def projetar_por_contrato(df: DataFrame, contrato: dict) -> DataFrame:
    view = registrar_view_sql(df, "projetar_contrato")
    return spark_sql(_select_contrato_sql(view, contrato, df.columns))


def projetar_contrato_hive(df: DataFrame, tabela: str) -> DataFrame:
    return projetar_por_contrato(df, CONTRATO_HIVE[tabela])


def projetar_contrato_oracle(df: DataFrame, tabela: str) -> DataFrame:
    return projetar_por_contrato(df, CONTRATO_ORACLE[tabela])


def validar_colunas_contrato(df: DataFrame, contrato: dict, contexto: str) -> None:
    exigir_colunas(df, nomes_campos_contrato_generico(contrato), contexto)


def registrar_erros_recomendacao(
    df_erros: DataFrame,
    logger_etapa,
    contexto: str,
    evento: str,
    motivo: str,
    colunas_detalhe: list[str] = None,
) -> None:
    if logger_etapa is None or df_erros is None:
        return

    if COL_NR_IDFR_PROJ not in df_erros.columns or COL_NR_IDFR_RCM not in df_erros.columns:
        colunas_tecnicas = [coluna for coluna in (colunas_detalhe or []) if coluna in df_erros.columns]
        if not colunas_tecnicas:
            logger_etapa.error(f"[{contexto}][{evento}] {motivo}")
            return

        view = registrar_view_sql(df_erros, "erros_tecnicos")
        linhas_tecnicas = coletar_sql(f"""
            SELECT DISTINCT {', '.join(coluna_sql(coluna) for coluna in colunas_tecnicas)}
            FROM {view}
        """)
        for linha in linhas_tecnicas:
            dados_linha = linha.asDict(recursive=True)
            detalhes = " ".join(
                f"{coluna}={dados_linha.get(coluna)}"
                for coluna in colunas_tecnicas
            )
            logger_etapa.error(f"[{contexto}][{evento}] {motivo} | {detalhes}")
        return

    colunas = [COL_NR_IDFR_PROJ, COL_NR_IDFR_RCM]
    opcionais = [COL_NR_VRS_VLDD_RCM, COL_CD_IDFR_AVS]
    for coluna in opcionais:
        if coluna in df_erros.columns and coluna not in colunas:
            colunas.append(coluna)

    for coluna in (colunas_detalhe or []):
        if coluna in df_erros.columns and coluna not in colunas:
            colunas.append(coluna)

    view = registrar_view_sql(df_erros, "erros_recomendacao")
    linhas = coletar_sql(f"""
        SELECT DISTINCT {', '.join(coluna_sql(coluna) for coluna in colunas)}
        FROM {view}
    """)

    for linha in linhas:
        dados_linha = linha.asDict(recursive=True)
        detalhe = {
            coluna.lower(): dados_linha[coluna]
            for coluna in colunas
            if coluna in dados_linha and coluna not in [COL_NR_IDFR_PROJ, COL_NR_IDFR_RCM, COL_NR_VRS_VLDD_RCM]
        }
        log_ciclo_recomendacao(
            logger_etapa,
            contexto,
            evento,
            linha[COL_NR_IDFR_PROJ],
            linha[COL_NR_IDFR_RCM],
            nr_vrs_vldd_rcm=dados_linha.get(COL_NR_VRS_VLDD_RCM),
            decisao="validacao bloqueou a continuidade da etapa",
            motivo=motivo,
            **detalhe,
        )


def validar_dominios_contrato(df: DataFrame, contrato: dict, contexto: str, logger_etapa=None) -> None:
    dominios = contrato.get("dominios", {})
    view = registrar_view_sql(df, "validar_dominios")

    for coluna, valores in dominios.items():
        if coluna not in df.columns:
            continue

        condicao_invalida = f"NOT ({coluna_sql(coluna)} IN ({lista_literais_sql(valores)}))"
        invalidos = existe_sql(f"""
            SELECT 1 AS ERRO
            FROM {view}
            WHERE {condicao_invalida}
        """)

        if invalidos:
            df_invalidos = spark_sql(f"""
                SELECT
                    *,
                    {literal_sql(coluna)} AS campo,
                    CAST({coluna_sql(coluna)} AS STRING) AS valor_encontrado,
                    {literal_sql(str(valores))} AS limite_esperado
                FROM {view}
                WHERE {condicao_invalida}
            """)
            registrar_erros_recomendacao(
                df_invalidos,
                logger_etapa,
                contexto,
                "DOMINIO_INVALIDO",
                f"coluna {coluna} possui valor fora do dominio",
                ["campo", "valor_encontrado", "limite_esperado"],
            )
            raise ErroContratoDados(
                codigo="CONTRATO_DOMINIO_INVALIDO",
                mensagem=f"{contexto}: coluna possui valor fora do dominio permitido.",
                detalhes={"coluna": coluna, "valores_permitidos": valores},
                acao="Corrigir os dados de origem ou o dominio definido no contrato.",
            )


def validar_sem_duplicidade(df: DataFrame, chaves: list[str], contexto: str, logger_etapa=None) -> None:
    exigir_colunas(df, chaves, contexto)
    view = registrar_view_sql(df, "validar_duplicidade")
    chaves_sql = ", ".join(coluna_sql(c) for c in chaves)

    duplicado = existe_sql(f"""
        SELECT 1 AS ERRO
        FROM {view}
        GROUP BY {chaves_sql}
        HAVING COUNT(1) > 1
    """)

    if duplicado:
        df_duplicados = spark_sql(f"""
            SELECT
                base.*,
                dup._qtd_duplicidade AS valor_encontrado,
                {literal_sql('1')} AS limite_esperado
            FROM {view} base
            INNER JOIN (
                SELECT {chaves_sql}, COUNT(1) AS _qtd_duplicidade
                FROM {view}
                GROUP BY {chaves_sql}
                HAVING COUNT(1) > 1
            ) dup
                ON {' AND '.join(f'base.{coluna_sql(chave)} <=> dup.{coluna_sql(chave)}' for chave in chaves)}
        """)
        registrar_erros_recomendacao(
            df_duplicados,
            logger_etapa,
            contexto,
            "DUPLICIDADE",
            f"duplicidade encontrada na chave {chaves}",
            chaves + ["valor_encontrado", "limite_esperado"],
        )
        raise ErroContratoDados(
            codigo="CONTRATO_CHAVE_DUPLICADA",
            mensagem=f"{contexto}: duplicidade encontrada na chave do contrato.",
            detalhes={"chaves": chaves},
            acao="Eliminar a duplicidade antes de continuar o processamento.",
        )


def validar_nao_nulo(df: DataFrame, colunas: list[str], contexto: str, logger_etapa=None) -> None:
    exigir_colunas(df, colunas, contexto)
    view = registrar_view_sql(df, "validar_nulos")

    for coluna in colunas:
        existe_nulo = existe_sql(f"""
            SELECT 1 AS ERRO
            FROM {view}
            WHERE {coluna_sql(coluna)} IS NULL
        """)

        if existe_nulo:
            df_invalidos = spark_sql(f"""
                SELECT
                    *,
                    {literal_sql(coluna)} AS campo,
                    CAST(NULL AS STRING) AS valor_encontrado,
                    {literal_sql('NAO_NULO')} AS limite_esperado
                FROM {view}
                WHERE {coluna_sql(coluna)} IS NULL
            """)
            registrar_erros_recomendacao(
                df_invalidos,
                logger_etapa,
                contexto,
                "NULO_OBRIGATORIO",
                f"coluna obrigatoria possui nulo: {coluna}",
                ["campo", "valor_encontrado", "limite_esperado"],
            )
            raise ErroContratoDados(
                codigo="CONTRATO_NULO_OBRIGATORIO",
                mensagem=f"{contexto}: coluna obrigatoria possui valor nulo.",
                detalhes={"coluna": coluna},
                acao="Preencher a coluna obrigatoria ou corrigir a origem dos dados.",
            )


def validar_contrato_hive(df: DataFrame, tabela: str, contexto: str, logger_etapa=None) -> None:
    contrato = CONTRATO_HIVE[tabela]
    validar_colunas_contrato(df, contrato, contexto)
    validar_nao_nulo(
        df,
        [nome_coluna for nome_coluna, _, aceita_nulo in contrato["campos"] if not aceita_nulo],
        contexto,
        logger_etapa=logger_etapa,
    )
    validar_dominios_contrato(df, contrato, contexto, logger_etapa=logger_etapa)


def validar_estrutura_ctl(df: DataFrame, contexto: str, logger_etapa=None) -> None:
    tabela = TABELAS_HIVE["hive_controle_operacional_recomendacao"]
    contrato = CONTRATO_HIVE[tabela]
    validar_contrato_hive(df, tabela, contexto, logger_etapa=logger_etapa)

    validar_nao_nulo(
        df,
        [
            COL_NR_IDFR_PROJ,
            COL_NR_IDFR_RCM,
            COL_NR_VRS_VLDD_RCM,
            COL_IN_REG_ATU,
            COL_IN_EXEA_PND,
            COL_TS_INC_REG,
        ],
        contexto,
        logger_etapa=logger_etapa,
    )

    view = registrar_view_sql(df, "validar_ctl")

    duplicado_atual = existe_sql(f"""
        SELECT 1 AS ERRO
        FROM {view}
        WHERE TRIM({coluna_sql(COL_IN_REG_ATU)}) = {literal_sql(VALOR_SIM)}
        GROUP BY {coluna_sql(COL_NR_IDFR_PROJ)}, {coluna_sql(COL_NR_IDFR_RCM)}
        HAVING COUNT(1) > 1
    """)

    if duplicado_atual:
        df_invalidos = spark_sql(f"""
            SELECT
                base.*,
                {literal_sql(COL_IN_REG_ATU)} AS campo,
                CAST(dup._qtd_atual AS STRING) AS valor_encontrado,
                {literal_sql('1')} AS limite_esperado
            FROM {view} base
            INNER JOIN (
                SELECT
                    {coluna_sql(COL_NR_IDFR_PROJ)},
                    {coluna_sql(COL_NR_IDFR_RCM)},
                    COUNT(1) AS _qtd_atual
                FROM {view}
                WHERE TRIM({coluna_sql(COL_IN_REG_ATU)}) = {literal_sql(VALOR_SIM)}
                GROUP BY {coluna_sql(COL_NR_IDFR_PROJ)}, {coluna_sql(COL_NR_IDFR_RCM)}
                HAVING COUNT(1) > 1
            ) dup
                ON base.{coluna_sql(COL_NR_IDFR_PROJ)} <=> dup.{coluna_sql(COL_NR_IDFR_PROJ)}
               AND base.{coluna_sql(COL_NR_IDFR_RCM)} <=> dup.{coluna_sql(COL_NR_IDFR_RCM)}
            WHERE TRIM(base.{coluna_sql(COL_IN_REG_ATU)}) = {literal_sql(VALOR_SIM)}
        """)
        registrar_erros_recomendacao(
            df_invalidos,
            logger_etapa,
            contexto,
            "CTL_ATUAL_DUPLICADA",
            "mais de uma linha atual por recomendacao",
            ["campo", "valor_encontrado", "limite_esperado"],
        )
        raise ErroContratoDados(
            codigo="CTL_MULTIPLA_LINHA_ATUAL",
            mensagem=f"{contexto}: mais de uma linha atual por recomendacao.",
            acao="Manter somente uma linha atual por projeto e recomendacao.",
        )

    atual_com_fim = existe_sql(f"""
        SELECT 1 AS ERRO
        FROM {view}
        WHERE TRIM({coluna_sql(COL_IN_REG_ATU)}) = {literal_sql(VALOR_SIM)}
          AND {coluna_sql(COL_TS_FIM_REG)} IS NOT NULL
    """)

    if atual_com_fim:
        df_invalidos = spark_sql(f"""
            SELECT
                *,
                {literal_sql(COL_TS_FIM_REG)} AS campo,
                CAST({coluna_sql(COL_TS_FIM_REG)} AS STRING) AS valor_encontrado,
                {literal_sql('NULL')} AS limite_esperado
            FROM {view}
            WHERE TRIM({coluna_sql(COL_IN_REG_ATU)}) = {literal_sql(VALOR_SIM)}
              AND {coluna_sql(COL_TS_FIM_REG)} IS NOT NULL
        """)
        registrar_erros_recomendacao(
            df_invalidos,
            logger_etapa,
            contexto,
            "CTL_ATUAL_COM_FIM",
            "linha atual com TS_FIM_REG preenchido",
            ["campo", "valor_encontrado", "limite_esperado"],
        )
        raise ErroContratoDados(
            codigo="CTL_ATUAL_COM_FIM",
            mensagem=f"{contexto}: linha atual possui TS_FIM_REG preenchido.",
            acao="Remover a data de fim da linha atual ou encerrá-la como historica.",
        )

    historico_sem_fim = existe_sql(f"""
        SELECT 1 AS ERRO
        FROM {view}
        WHERE TRIM({coluna_sql(COL_IN_REG_ATU)}) = {literal_sql(VALOR_NAO)}
          AND {coluna_sql(COL_TS_FIM_REG)} IS NULL
    """)

    if historico_sem_fim:
        df_invalidos = spark_sql(f"""
            SELECT
                *,
                {literal_sql(COL_TS_FIM_REG)} AS campo,
                CAST({coluna_sql(COL_TS_FIM_REG)} AS STRING) AS valor_encontrado,
                {literal_sql('NAO_NULO')} AS limite_esperado
            FROM {view}
            WHERE TRIM({coluna_sql(COL_IN_REG_ATU)}) = {literal_sql(VALOR_NAO)}
              AND {coluna_sql(COL_TS_FIM_REG)} IS NULL
        """)
        registrar_erros_recomendacao(
            df_invalidos,
            logger_etapa,
            contexto,
            "CTL_HISTORICO_SEM_FIM",
            "linha historica sem TS_FIM_REG",
            ["campo", "valor_encontrado", "limite_esperado"],
        )
        raise ErroContratoDados(
            codigo="CTL_HISTORICO_SEM_FIM",
            mensagem=f"{contexto}: linha historica nao possui TS_FIM_REG.",
            acao="Informar a data de encerramento da linha historica.",
        )

    fim_menor_inicio = existe_sql(f"""
        SELECT 1 AS ERRO
        FROM {view}
        WHERE {coluna_sql(COL_TS_FIM_REG)} IS NOT NULL
          AND {coluna_sql(COL_TS_FIM_REG)} < {coluna_sql(COL_TS_INC_REG)}
    """)

    if fim_menor_inicio:
        df_invalidos = spark_sql(f"""
            SELECT
                *,
                {literal_sql(COL_TS_FIM_REG)} AS campo,
                CAST({coluna_sql(COL_TS_FIM_REG)} AS STRING) AS valor_encontrado,
                CONCAT('>= ', CAST({coluna_sql(COL_TS_INC_REG)} AS STRING)) AS limite_esperado
            FROM {view}
            WHERE {coluna_sql(COL_TS_FIM_REG)} IS NOT NULL
              AND {coluna_sql(COL_TS_FIM_REG)} < {coluna_sql(COL_TS_INC_REG)}
        """)
        registrar_erros_recomendacao(
            df_invalidos,
            logger_etapa,
            contexto,
            "CTL_FIM_MENOR_INICIO",
            "TS_FIM_REG menor que TS_INC_REG",
            ["campo", "valor_encontrado", "limite_esperado"],
        )
        raise ErroContratoDados(
            codigo="CTL_PERIODO_INVALIDO",
            mensagem=f"{contexto}: TS_FIM_REG e menor que TS_INC_REG.",
            acao="Corrigir o periodo operacional da linha de controle.",
        )


def dataframe_vazio_por_contrato_hive(tabela: str) -> DataFrame:
    contrato = CONTRATO_HIVE[tabela]
    expressoes = [
        f"CAST(NULL AS {tipo_sql(tipo_coluna)}) AS {coluna_sql(nome_coluna)}"
        for nome_coluna, tipo_coluna, _ in contrato["campos"]
    ]
    return spark_sql(f"SELECT {', '.join(expressoes)} WHERE 1 = 0")


def dataframe_por_linhas_sql(linhas: list[dict], tabela: str) -> DataFrame:
    contrato = CONTRATO_HIVE[tabela]

    if not linhas:
        return dataframe_vazio_por_contrato_hive(tabela)

    selects = []
    for linha in linhas:
        expressoes = []
        for nome_coluna, tipo_coluna, _ in contrato["campos"]:
            expressoes.append(
                f"{literal_sql_tipado(linha.get(nome_coluna), tipo_coluna)} AS {coluna_sql(nome_coluna)}"
            )
        selects.append("SELECT " + ", ".join(expressoes))

    return spark_sql("\nUNION ALL\n".join(selects))


def unir_dataframes_por_nome_sql(dataframes: list[DataFrame], colunas: list[str] = None) -> DataFrame:
    dfs = [df for df in dataframes if df is not None]

    if not dfs:
        raise ValueError("Nenhum DataFrame informado para uniao SQL.")

    colunas_finais = colunas or list(dfs[0].columns)
    partes = []

    for indice, df in enumerate(dfs):
        exigir_colunas(df, colunas_finais, f"UNIAO_SQL_{indice}")
        view = registrar_view_sql(df, f"uniao_sql_{indice}")
        selecao = ", ".join(coluna_sql(coluna) for coluna in colunas_finais)
        partes.append(f"SELECT {selecao} FROM {view}")

    return spark_sql("\nUNION ALL\n".join(partes))


def dataframe_tem_registros(df: DataFrame) -> bool:
    return existe_dataframe_sql(df, "DATAFRAME_TEM_REGISTROS")


def tipo_origem(nm_db_ogm: str) -> str:
    valor = (nm_db_ogm or "").strip().lower()

    if any(valor.startswith(prefixo) for prefixo in PREFIXOS_ORIGEM_HIVE):
        return TIPO_ORIGEM_HIVE

    if any(valor.startswith(prefixo) for prefixo in PREFIXOS_ORIGEM_DB2):
        return TIPO_ORIGEM_DB2

    return TIPO_ORIGEM_DESCONHECIDA


def colunas_necessarias_origem(linha_ctl=None) -> list[str]:
    colunas = list(COLUNAS_CLIENTE_ORIGEM) + list(COLUNAS_CANDIDATAS_DT_EXEC_FONTE)

    if linha_ctl:
        coluna_escopo = (linha_ctl.get(COL_NM_COL_ECP_OGM) or "").strip()
        if coluna_escopo:
            colunas.append(coluna_escopo)

    colunas_unicas = []
    vistos = set()

    for coluna in colunas:
        coluna_limpa = (coluna or "").strip()
        chave = coluna_limpa.lower()
        if coluna_limpa and chave not in vistos:
            colunas_unicas.append(coluna_limpa)
            vistos.add(chave)

    return colunas_unicas


def colunas_existentes_db2(
    cliente_db2,
    schema: str,
    tabela: str,
    colunas_candidatas: list[str],
) -> list[str]:
    tabela_catalogo = "SYSIBM.SYSCOLUMNS"
    campo_schema = "TBCREATOR"
    campo_tabela = "TBNAME"
    campo_coluna = "NAME"
    campo_ordem = "COLNO"
    alias_coluna = "NOME_COLUNA"

    schema_sql = validar_identificador_sql(schema, 'schema_db2').upper()
    tabela_sql = validar_identificador_sql(tabela, 'tabela_db2').upper()
    colunas_normalizadas = [
        validar_identificador_sql(coluna, 'coluna_origem').upper()
        for coluna in colunas_candidatas
        if (coluna or "").strip()
    ]

    if not colunas_normalizadas:
        return []

    lista_colunas = ", ".join(literal_sql(coluna) for coluna in colunas_normalizadas)
    query = f"""
        SELECT
            {campo_coluna} AS {alias_coluna}
        FROM {tabela_catalogo}
        WHERE {campo_schema} = {literal_sql(schema_sql)}
          AND {campo_tabela} = {literal_sql(tabela_sql)}
          AND UPPER({campo_coluna}) IN ({lista_colunas})
        ORDER BY {campo_ordem}
    """

    logger.info(
        f"[DB2_ZOS][COLUNAS_EXISTENTES][INICIO] schema={schema_sql} "
        f"tabela={tabela_sql} colunas_candidatas={colunas_normalizadas} "
        f"catalogo={tabela_catalogo}"
    )
    linhas = coletar_dataframe_sql(cliente_db2.run_select(query), "COLUNAS_DB2")
    colunas_encontradas = [linha[alias_coluna] for linha in linhas]
    logger.info(
        f"[DB2_ZOS][COLUNAS_EXISTENTES][FIM] schema={schema_sql} "
        f"tabela={tabela_sql} colunas_encontradas={colunas_encontradas}"
    )
    return colunas_encontradas


def ler_origem_por_metadado(
    nm_db_ogm: str,
    nm_tab_ogm: str,
    cliente_db2=None,
    colunas_origem=None,
) -> DataFrame:
    db = (nm_db_ogm or "").strip()
    tabela = (nm_tab_ogm or "").strip()

    if not db or not tabela:
        raise ValueError("NM_DB_OGM e NM_TAB_OGM sao obrigatorios para ler origem.")

    origem = tipo_origem(db)

    if origem == TIPO_ORIGEM_HIVE:
        return spark_sql(f"SELECT * FROM {nome_qualificado_spark_sql(db, tabela)}")

    if origem == TIPO_ORIGEM_DB2:
        if cliente_db2 is None:
            raise ValueError("cliente_db2 obrigatorio para origem DB2.")

        colunas_candidatas = colunas_origem or colunas_necessarias_origem()
        logger.info(
            f"[DB2_ZOS][LER_ORIGEM][DB2] db={db} tabela={tabela} "
            f"colunas_candidatas={colunas_candidatas}"
        )
        colunas_existentes = colunas_existentes_db2(
            cliente_db2,
            db,
            tabela,
            colunas_candidatas,
        )
        logger.info(
            f"[DB2_ZOS][LER_ORIGEM][COLUNAS_RESOLVIDAS] db={db} tabela={tabela} "
            f"colunas_existentes={colunas_existentes}"
        )
        nome_tabela = nome_qualificado_sql(db, tabela)

        if not colunas_existentes:
            coluna_linha_tecnica = "_LINHA_ORIGEM"
            valor_linha_tecnica = 1
            query = f"""
                SELECT
                    {valor_linha_tecnica} AS {coluna_linha_tecnica}
                FROM {nome_tabela}
            """
        else:
            colunas_sql = ",\n                    ".join(
                validar_identificador_sql(coluna, 'coluna_origem_db2')
                for coluna in colunas_existentes
            )
            query = f"""
                SELECT
                    {colunas_sql}
                FROM {nome_tabela}
            """

        return cliente_db2.run_select(query)

    raise ValueError(f"Tipo de origem desconhecida para NM_DB_OGM={nm_db_ogm}.")


def aplicar_escopo_origem(df_origem: DataFrame, linha_ctl: dict) -> DataFrame:
    tipo_escopo = linha_ctl.get(COL_CD_TIP_ECP_OGM)

    if tipo_escopo == TIPO_ESCOPO_ORIGEM_INTEIRA:
        return df_origem

    if tipo_escopo != TIPO_ESCOPO_ORIGEM_SUBCONJUNTO:
        raise ValueError(f"CD_TIP_ECP_OGM invalido: {tipo_escopo}")

    coluna_escopo = (linha_ctl.get(COL_NM_COL_ECP_OGM) or "").strip()
    valor_escopo = linha_ctl.get(COL_NR_IDFR_PBCO)

    if not coluna_escopo:
        raise ValueError("NM_COL_ECP_OGM obrigatorio para CD_TIP_ECP_OGM=2.")

    if valor_escopo is None:
        raise ValueError("NR_IDFR_PBCO obrigatorio para CD_TIP_ECP_OGM=2.")

    coluna_real = resolver_coluna_case_insensitive(df_origem, [coluna_escopo])

    if not coluna_real:
        raise ValueError(f"Coluna de escopo inexistente na origem: {coluna_escopo}")

    tabela_ctl = TABELAS_HIVE["hive_controle_operacional_recomendacao"]
    tipo_nr_idfr_pbco = tipo_campo_contrato_hive(tabela_ctl, COL_NR_IDFR_PBCO)
    view = registrar_view_sql(df_origem, "escopo_origem")

    return spark_sql(f"""
        SELECT *
        FROM {view}
        WHERE CAST({coluna_sql(coluna_real)} AS {tipo_sql(tipo_nr_idfr_pbco)}) =
              {literal_sql_tipado(valor_escopo, tipo_nr_idfr_pbco)}
    """)


def resolver_referencia_origem(df_origem: DataFrame) -> dict:
    coluna_data = resolver_coluna_case_insensitive(
        df_origem,
        COLUNAS_CANDIDATAS_DT_EXEC_FONTE,
    )
    view = registrar_view_sql(df_origem, "referencia_origem")

    if coluna_data:
        linhas = coletar_sql(f"""
            SELECT MAX(TO_TIMESTAMP({coluna_sql(coluna_data)})) AS max_dt
            FROM {view}
        """)
        max_dt = linhas[0]["max_dt"]

        if max_dt is not None:
            return {
                "forma": FORMA_REFERENCIA_ORIGEM_DATA,
                "coluna_data": coluna_data,
                "ts_observado": max_dt,
                "quantidade": None,
            }

    quantidade = contar_dataframe_sql(df_origem, "REFERENCIA_ORIGEM")
    return {
        "forma": FORMA_REFERENCIA_ORIGEM_QUANTIDADE,
        "coluna_data": None,
        "ts_observado": None,
        "quantidade": quantidade,
    }


def resolver_clientes_origem(df_origem: DataFrame) -> DataFrame:
    coluna_cliente = resolver_coluna_case_insensitive(df_origem, COLUNAS_CLIENTE_ORIGEM)

    if not coluna_cliente:
        raise ValueError(
            "Origem sem coluna de cliente aceita. "
            f"Esperado uma de {COLUNAS_CLIENTE_ORIGEM}."
        )

    tabela_evtl = TABELAS_HIVE["hive_cliente_recomendacao_historica"]
    tipo_cd_cli = tipo_campo_contrato_hive(tabela_evtl, COL_CD_CLI)
    tipo_texto = tipo_campo_contrato_hive(tabela_evtl, COL_CD_IDFR_AVS)
    view = registrar_view_sql(df_origem, "clientes_origem")

    return spark_sql(f"""
        WITH normalizado AS (
            SELECT TRIM(CAST({coluna_sql(coluna_cliente)} AS {tipo_sql(tipo_texto)})) AS _cd_cli_raw
            FROM {view}
        ), convertido AS (
            SELECT CAST(_cd_cli_raw AS {tipo_sql(tipo_cd_cli)}) AS {coluna_sql(COL_CD_CLI)}
            FROM normalizado
            WHERE _cd_cli_raw IS NOT NULL
              AND _cd_cli_raw <> ''
        )
        SELECT DISTINCT {coluna_sql(COL_CD_CLI)}
        FROM convertido
        WHERE {coluna_sql(COL_CD_CLI)} IS NOT NULL
    """)


# def cd_idfr_avs_sql_expr(alias: str = None) -> str:
#     return (
#         f"CONCAT(CAST({coluna_sql(COL_NR_IDFR_PROJ, alias)} AS STRING), "
#         f"'_', CAST({coluna_sql(COL_NR_IDFR_RCM, alias)} AS STRING))"
#     )

def cd_idfr_avs_sql_expr(alias: str = None) -> str:
    return f"CAST({coluna_sql(COL_NR_IDFR_RCM, alias)} AS STRING)"

# def cd_idfr_avs_valor(nr_idfr_proj, nr_idfr_rcm) -> str:
#     return f"{nr_idfr_proj}_{nr_idfr_rcm}"

def cd_idfr_avs_valor(nr_idfr_proj, nr_idfr_rcm) -> str:
    return f"{nr_idfr_rcm}"


def log_ciclo_recomendacao(
    logger_etapa,
    etapa: str,
    evento: str,
    nr_idfr_proj,
    nr_idfr_rcm,
    nr_vrs_vldd_rcm=None,
    decisao: str = None,
    motivo: str = None,
    **detalhes,
) -> None:
    partes = [
        f"[{etapa}][CICLO_VIDA][{evento}]",
        f"Recomendacao nr_idfr_proj={nr_idfr_proj} nr_idfr_rcm={nr_idfr_rcm}",
    ]

    if nr_vrs_vldd_rcm is not None:
        partes.append(f"nr_vrs_vldd_rcm={nr_vrs_vldd_rcm}")

    if decisao:
        partes.append(f"decisao={decisao}")

    if motivo:
        partes.append(f"motivo={motivo}")

    for nome, valor in detalhes.items():
        if valor is not None:
            partes.append(f"{nome}={valor}")

    logger_etapa.info(" | ".join(partes))


def preparar_append_idempotente_evtl(
    df_lote: DataFrame,
    df_historico: DataFrame,
    logger_etapa=None,
) -> DataFrame:
    tabela_evtl = TABELAS_HIVE["hive_cliente_recomendacao_historica"]
    chaves = CONTRATO_HIVE[tabela_evtl]["chaves"]
    colunas = nomes_campos_contrato(tabela_evtl)

    df_lote = projetar_contrato_hive(df_lote, tabela_evtl)
    df_historico = projetar_contrato_hive(df_historico, tabela_evtl)

    validar_contrato_hive(df_lote, tabela_evtl, "RCM_FNC_CLI_EVTL_LOTE", logger_etapa=logger_etapa)
    validar_contrato_hive(df_historico, tabela_evtl, "RCM_FNC_CLI_EVTL_HISTORICO", logger_etapa=logger_etapa)
    validar_sem_duplicidade(df_lote, chaves, "RCM_FNC_CLI_EVTL_LOTE", logger_etapa=logger_etapa)
    validar_sem_duplicidade(df_historico, chaves, "RCM_FNC_CLI_EVTL_HISTORICO", logger_etapa=logger_etapa)

    view_lote = registrar_view_sql(df_lote, "evtl_lote")
    view_hist = registrar_view_sql(df_historico, "evtl_hist")
    cond_chaves = " AND ".join(
        f"novo.{coluna_sql(chave)} <=> hist.{coluna_sql(chave)}"
        for chave in chaves
    )
    colunas_nao_chave = [c for c in colunas if c not in chaves]
    cond_divergencias = " OR ".join(
        f"NOT (novo.{coluna_sql(c)} <=> hist.{coluna_sql(c)})"
        for c in colunas_nao_chave
    )

    if cond_divergencias:
        existe_divergencia = existe_sql(f"""
            SELECT 1 AS ERRO
            FROM {view_lote} novo
            INNER JOIN {view_hist} hist
                ON {cond_chaves}
            WHERE {cond_divergencias}
        """)

        if existe_divergencia:
            df_divergentes = spark_sql(f"""
                SELECT
                    novo.*,
                    {literal_sql('CONTEUDO')} AS campo,
                    {literal_sql('DIVERGENTE')} AS valor_encontrado,
                    {literal_sql('MESMO_CONTEUDO_PARA_CHAVE_EXISTENTE')} AS limite_esperado
                FROM {view_lote} novo
                INNER JOIN {view_hist} hist
                    ON {cond_chaves}
                WHERE {cond_divergencias}
            """)
            registrar_erros_recomendacao(
                df_divergentes,
                logger_etapa,
                "RCM_FNC_CLI_EVTL_APPEND_IDEMPOTENTE",
                "CONTEUDO_DIVERGENTE",
                "mesma chave historica com conteudo divergente",
                chaves + ["campo", "valor_encontrado", "limite_esperado"],
            )
            raise ValueError(
                "RCM_FNC_CLI_EVTL possui mesma chave historica com conteudo divergente."
            )

    selecao_lote = ", ".join(f"novo.{coluna_sql(c)}" for c in colunas)
    cond_anti = " AND ".join(
        f"novo.{coluna_sql(chave)} <=> hist.{coluna_sql(chave)}"
        for chave in chaves
    )

    return spark_sql(f"""
        SELECT {selecao_lote}
        FROM {view_lote} novo
        LEFT ANTI JOIN (
            SELECT DISTINCT {', '.join(coluna_sql(chave) for chave in chaves)}
            FROM {view_hist}
        ) hist
            ON {cond_anti}
    """)


def comparar_dataframes_materialmente(
    df_esquerda: DataFrame,
    df_direita: DataFrame,
    colunas: list[str],
    contrato: dict,
) -> bool:
    tipos = {
        nome_coluna: tipo_coluna
        for nome_coluna, tipo_coluna, _
        in contrato["campos"]
    }
    faltantes = [coluna for coluna in colunas if coluna not in tipos]

    if faltantes:
        raise ValueError(f"Colunas sem tipo no contrato para comparacao material: {faltantes}")

    view_esquerda = registrar_view_sql(df_esquerda, "comparar_esquerda")
    view_direita = registrar_view_sql(df_direita, "comparar_direita")
    proj_esquerda = ", ".join(
        f"CAST({coluna_sql(c)} AS {tipo_sql(tipos[c])}) AS {coluna_sql(c)}"
        for c in colunas
    )
    proj_direita = ", ".join(
        f"CAST({coluna_sql(c)} AS {tipo_sql(tipos[c])}) AS {coluna_sql(c)}"
        for c in colunas
    )

    diff_1 = existe_sql(f"""
        WITH esquerda AS (
            SELECT {proj_esquerda}
            FROM {view_esquerda}
        ), direita AS (
            SELECT {proj_direita}
            FROM {view_direita}
        )
        SELECT 1 AS DIFERENCA
        FROM (
            SELECT * FROM esquerda
            EXCEPT ALL
            SELECT * FROM direita
        ) diff
    """)

    if diff_1:
        return True

    diff_2 = existe_sql(f"""
        WITH esquerda AS (
            SELECT {proj_esquerda}
            FROM {view_esquerda}
        ), direita AS (
            SELECT {proj_direita}
            FROM {view_direita}
        )
        SELECT 1 AS DIFERENCA
        FROM (
            SELECT * FROM direita
            EXCEPT ALL
            SELECT * FROM esquerda
        ) diff
    """)
    return bool(diff_2)


def ultima_instancia_historica_evtl(df_evtl: DataFrame) -> DataFrame:
    tabela_evtl = TABELAS_HIVE["hive_cliente_recomendacao_historica"]
    df_evtl = projetar_contrato_hive(df_evtl, tabela_evtl)
    view = registrar_view_sql(df_evtl, "ultima_evtl")
    colunas = nomes_campos_contrato(tabela_evtl)
    selecao = ", ".join(f"e.{coluna_sql(coluna)} AS {coluna_sql(coluna)}" for coluna in colunas)

    return spark_sql(f"""
        WITH max_data AS (
            SELECT
                {coluna_sql(COL_CD_IDFR_AVS)},
                MAX({coluna_sql(COL_DT_AVS_FNC_CLI)}) AS _dt_max
            FROM {view}
            GROUP BY {coluna_sql(COL_CD_IDFR_AVS)}
        )
        SELECT {selecao}
        FROM {view} e
        INNER JOIN max_data m
            ON e.{coluna_sql(COL_CD_IDFR_AVS)} <=> m.{coluna_sql(COL_CD_IDFR_AVS)}
           AND e.{coluna_sql(COL_DT_AVS_FNC_CLI)} <=> m._dt_max
    """)

